# Day 11 — SQL Basics: Asking Questions of a Database

---

## Why SQL matters

Every data job — analyst, scientist, engineer, PropTech researcher — requires SQL. It is the language you use to talk to databases, which is where most real-world data actually lives (not in CSV files).

The good news: SQL reads like English. You are going to write sentences like:

> *"Give me the average sale price, grouped by street name, only for buildings with more than 2 units, ordered from most expensive to least."*

That sentence is almost literally valid SQL.

## The math-teacher analogy

In pandas you've been doing things like:
```python
df[df['unitsres'] > 2].groupby('STREET NAME')['DOC. AMOUNT'].mean().sort_values(ascending=False)
```

SQL expresses the same thought differently — the pieces are the same, just in a different order:
```sql
SELECT "STREET NAME", AVG("DOC. AMOUNT")
FROM sales
WHERE unitsres > 2
GROUP BY "STREET NAME"
ORDER BY AVG("DOC. AMOUNT") DESC;
```

Today you will re-run Day 10's analysis — same Bushwick data, same questions — but in SQL. Seeing both approaches side by side is the fastest way to learn SQL if you already know pandas.

## Today's question

Using SQL, answer: **which Bushwick streets have the highest median sale price, and how does that vary by building size?**

---

## Organization rules
- One task per cell.
- Sanity check after every major step.
- Every SQL query gets a one-sentence plain-English description above it.

In [39]:
# CELL 1 — Imports only.
# sqlite3 is Python's built-in SQL engine — no install needed.
# Think of it as a lightweight database that lives in memory or in a file.
import sqlite3
import pandas as pd

In [40]:
# CELL 2 — Load CSVs into pandas, then push them into a SQLite database.
#
# New concept: conn = sqlite3.connect(':memory:')
# This creates a database that lives in RAM, not on disk.
# ':memory:' means "don't save a file, just hold this for now."
#
# df.to_sql('table_name', conn) writes a DataFrame into the database
# as a table you can query with SQL.
#
# TODO:
#   1. Load acris_bushwick_clean.csv into df_sales
#   2. Load pluto.csv into df_pluto, filter to community board == 304.0
#   3. Create conn = sqlite3.connect(':memory:')
#   4. Write df_sales into conn as a table called 'sales'
#      Use: df_sales.to_sql('sales', conn, index=False, if_exists='replace')
#   5. Write the filtered df_pluto into conn as a table called 'pluto'
#   6. Sanity check: print('Tables loaded')

df_sales = pd.read_csv('../data/acris_bushwick_clean.csv')
df_pluto = pd.read_csv('../data/pluto.csv')

df_pluto = df_pluto[df_pluto['community board'] == 304]

conn = sqlite3.connect(':memory:')
df_sales.to_sql('sales', conn, index=False, if_exists='replace')

df_pluto.to_sql('pluto', conn, index=False, if_exists='replace')

print('Tables loaded')
 

/var/folders/01/7mmm8p_55mz_znpc8yd21gjc0000gn/T/ipykernel_78422/3511738355.py:20: DtypeWarning: Columns (0: zonedist3, 1: zonedist4, 2: overlay2, 3: spdist1, 4: spdist2, 5: ltdheight, 6: splitzone, 7: unitsres, 8: unitstotal, 9: lotfront, 10: lotdepth, 11: bldgfront, 12: bldgdepth, 13: irrlotcode, 14: histdist, 15: landmark, 16: condono, 17: edesignum, 18: dcpedited, 19: mihopt1, 20: mihopt2, 21: mihopt3, 22: mihopt4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pluto = pd.read_csv('../data/pluto.csv')


Tables loaded


## The 5 SQL keywords you need today

| Keyword | What it does | Pandas equivalent |
|---|---|---|
| `SELECT` | Choose which columns to return | `df[['col1', 'col2']]` |
| `FROM` | Which table to query | The DataFrame name |
| `WHERE` | Filter rows | `df[df['col'] > value]` |
| `GROUP BY` | Group rows for aggregation | `.groupby('col')` |
| `ORDER BY` | Sort the result | `.sort_values()` |

You always write them in this order: SELECT → FROM → WHERE → GROUP BY → ORDER BY.

To run a SQL query in Python and get a DataFrame back:
```python
result = pd.read_sql("SELECT * FROM sales", conn)
```

In [41]:
# CELL 3 — Your first query: SELECT all rows from sales.
# Plain English: "Show me everything in the sales table."
#
# TODO: write a pd.read_sql() call that runs SELECT * FROM sales
# and prints the shape of the result.
# It should match the shape of df_sales.


result = pd.read_sql("SELECT * FROM sales", conn)

print(result)


         DOCUMENT ID RECORD TYPE_x  BOROUGH_x  BLOCK   LOT EASEMENT  \
0   2026042900367001             L          3   3221    60        N   
1   2025091000394001             L          3   3406    57        N   
2   2025081500408001             L          3   3250     8        N   
3   2024080600216001             L          3   3370   150        N   
4   2024123000495001             L          3   3418    40        N   
5   2025011000293001             L          3   3402  1015        N   
6   2024121300394001             L          3   3187    27        N   
7   2024081300075001             L          3   3432    14        N   
8   2024071700143001             L          3   3397    60        N   
9   2024031100249001             L          3   3226    29        N   
10  2023110700697001             L          3   3243  1005        N   
11  2023102500023001             L          3   3263  1206        N   
12  2023101800162001             L          3   3378    37        N   
13  20

In [42]:
# CELL 4 — WHERE: filter to sales above $500,000.
# Plain English: "Show me all sales where the price was over $500k."
#
# TODO: write a query using WHERE "DOC. AMOUNT" > 500000
# Print the result length and the first 5 rows.
#
# Note: column names with spaces or punctuation need double quotes in SQL.
# "DOC. AMOUNT" not DOC. AMOUNT

# Query to filter rows where doc_amount is greater than 500,000
query = 'SELECT * FROM sales WHERE "DOC. AMOUNT" > 500000'



result = pd.read_sql("SELECT * FROM sales", conn)

result = pd.read_sql(query, conn)

print(result)





         DOCUMENT ID RECORD TYPE_x  BOROUGH_x  BLOCK   LOT EASEMENT  \
0   2026042900367001             L          3   3221    60        N   
1   2025081500408001             L          3   3250     8        N   
2   2024080600216001             L          3   3370   150        N   
3   2024123000495001             L          3   3418    40        N   
4   2024121300394001             L          3   3187    27        N   
5   2023110700697001             L          3   3243  1005        N   
6   2023102500023001             L          3   3263  1206        N   
7   2023101800162001             L          3   3378    37        N   
8   2023091500691001             L          3   3357  1105        N   
9   2023071400027001             L          3   3296     3        N   
10  2023052200938001             L          3   3387  1110        N   
11  2023041600002001             L          3   3279    25        N   
12  2023032200907001             L          3   3132  1305        N   
13  20

In [46]:
# CELL 5 — GROUP BY: median sale price by street name.
# Plain English: "For each street, give me the median sale price, sorted high to low."
#
# New concept: SQL uses AVG(), COUNT(), SUM(), MIN(), MAX() for aggregation.
# There is no built-in MEDIAN() in SQLite — use AVG() for now as an approximation.
#
# TODO:
#   SELECT "STREET NAME", AVG("DOC. AMOUNT") AS avg_price, COUNT(*) AS num_sales
#   FROM sales
#   GROUP BY "STREET NAME"
#   ORDER BY avg_price DESC
#
# Print the result. Do the streets match what you saw in Day 10?

query = """
SELECT "STREET NAME", AVG("DOC. AMOUNT") AS avg_price FROM sales 
WHERE "DOC. AMOUNT" > 500000
GROUP BY "STREET NAME"
ORDER BY avg_price DESC

"""

result = pd.read_sql(query, conn)
print(result)



           STREET NAME  avg_price
0          HART STREET  1825000.0
1   ST NICHOLAS AVENUE  1800000.0
2      TROUTMAN STREET  1750000.0
3       WYCKOFF AVENUE  1600000.0
4         GATES AVENUE  1265000.0
5      BUSHWICK AVENUE  1234119.0
6        PUTNAM AVENUE  1190000.0
7        IRVING AVENUE  1150000.0
8       MADISON STREET  1100000.0
9            EVERGREEN  1100000.0
10         CENTRAL AVE  1048888.0
11       HIMROD STREET   920000.0
12       COVERT STREET   880000.0
13    EVERGREEN AVENUE   779000.0
14       HALSEY STREET   767500.0
15    JEFFERSON AVENUE   755000.0
16      HANCOCK STREET   740000.0
17       KOSSUTH PLACE   635000.0
18       HARMAN STREET   575000.0
19      FAYETTE STREET   550000.0
20       DEKALB AVENUE   540000.0
21       LAFAYETTE AVE   529000.0
22      DECATUR STREET   510000.0


**Before moving on — answer these:**
- How does `GROUP BY` in SQL compare to `.groupby()` in pandas? What's the same, what's different?
- Why did we use `AVG()` instead of `MEDIAN()`? What does that mean for our results?
- What does `COUNT(*)` count — rows, or non-null values?

um when I used groupby i had to call the dataframe and the specific category everytime I don't have to do that here

SQL doesnt have an median method so I called average but aversage can be swayed higher due to higher dollar amounts

count counts the number of rows in a data set

In [51]:
# CELL 6 — JOIN: connect sales to pluto on block and lot.
# Plain English: "For each sale, look up the building details from PLUTO."
#
# SQL JOIN is the same idea as pd.merge() — match rows from two tables
# on a shared key.
#
# Syntax:
#   SELECT s."STREET NAME", s."DOC. AMOUNT", p.unitsres, p.bldgarea
#   FROM sales s
#   JOIN pluto p
#     ON s.BLOCK = p."Tax block"
#     AND s.LOT   = p."Tax lot"
#     AND s.BOROUGH_x = p.borocode
#
# The letters after table names (s, p) are aliases — shortcuts so you
# don't have to type the full table name every time.
#
# TODO: write and run the JOIN query above.
# Store as df_joined. Print len(df_joined) — should match Day 10's df_merged.

df_joined = """
SELECT s."STREET NAME", s."DOC. AMOUNT", p.unitsres, p.bldgarea
FROM sales s
JOIN pluto p
    ON s.BLOCK = p."Tax block"
    AND s.LOT = p."Tax lot"
    AND s.BOROUGH_x = p.borocode
"""
result = pd.read_sql(df_joined, conn)
print(result)




           STREET NAME  DOC. AMOUNT unitsres bldgarea
0          HART STREET    1825000.0        1    1,452
1        HALSEY STREET     300000.0        8    6,890
2   ST NICHOLAS AVENUE    1800000.0        6    4,725
3        PUTNAM AVENUE    1190000.0        2    1,960
4        COVERT STREET     880000.0        3    3,300
5      TROUTMAN STREET    1750000.0        0    2,500
6       DECATUR STREET     447800.0        3    3,207
7     WEIRFIELD STREET      75000.0        2    2,700
8         CEDAR STREET      35000.0        2    2,074
9        IRVING AVENUE    1150000.0        5    3,300
10           EVERGREEN    1100000.0        2    1,876
11    STOCKHOLM STREET      25000.0        3    3,300
12      DECATUR STREET      25000.0        2    2,400
13       HIMROD STREET     920000.0        6    4,875
14       GREENE AVENUE     200000.0        2      800
15        GATES AVENUE     130000.0        2    2,100
16       DEKALB AVENUE     150000.0        3    1,800
17    JEFFERSON AVENUE     7

In [61]:
# CELL 7 — price per unit in SQL.
# Plain English: "For each sale matched to PLUTO, calculate price per unit
#                and show me the street name and building size."
#
# You can do math directly inside SELECT:
#   SELECT "STREET NAME", "DOC. AMOUNT" / unitsres AS price_per_unit
#
# TODO:
#   Build on the JOIN from Cell 6.
#   Add a WHERE clause: unitsres > 0 AND unitsres < 100
#   Add the price_per_unit calculation in SELECT.
#   ORDER BY price_per_unit DESC.
#   Print the top 10 rows.

df_joined = """
SELECT s."STREET NAME", s."DOC. AMOUNT", p.unitsres, p.bldgarea, s."DOC. AMOUNT" / CAST(p.unitsres AS INTEGER) AS price_per_unit
FROM sales s
JOIN pluto p
    ON s.BLOCK = p."Tax block"
    AND s.LOT = p."Tax lot"
    AND s.BOROUGH_x = p.borocode
    WHERE CAST (p.unitsres AS INTEGER) > 0 AND CAST (p.unitsres AS INTEGER) < 100
    ORDER BY price_per_unit DESC
"""
result = pd.read_sql(df_joined, conn)

pd.options.display.float_format = '${:,.0f}'.format
print(result.to_string())





           STREET NAME  DOC. AMOUNT unitsres bldgarea  price_per_unit
0          HART STREET   $1,825,000        1    1,452      $1,825,000
1         GATES AVENUE   $1,265,000        2    1,800        $632,500
2        PUTNAM AVENUE   $1,190,000        2    1,960        $595,000
3            EVERGREEN   $1,100,000        2    1,876        $550,000
4          CENTRAL AVE   $1,048,888        2    1,967        $524,444
5     JEFFERSON AVENUE     $725,000        2    1,960        $362,500
6        HALSEY STREET     $725,000        2    2,430        $362,500
7       WYCKOFF AVENUE   $1,600,000        5    4,500        $320,000
8   ST NICHOLAS AVENUE   $1,800,000        6    4,725        $300,000
9        COVERT STREET     $880,000        3    3,300        $293,333
10       HALSEY STREET     $810,000        3    2,988        $270,000
11       IRVING AVENUE   $1,150,000        5    3,300        $230,000
12       HIMROD STREET     $920,000        6    4,875        $153,333
13      DECATUR STRE

**Critical thinking pause:**
- You just did in SQL what took most of Day 10 in pandas. What felt easier? What felt harder?
- In pandas, you had to debug column types (int vs float vs string) repeatedly. Does SQL have the same problem, or does it handle types differently?
- When would you choose SQL over pandas? When would you choose pandas over SQL?

SQL felt a lot easier it was easier to call the methods and combine data sets with sql

I actually had to convert some of this into ints from strings but it was a lot easier tha doing so with python

When i need to sift through and clean data quickly i would probably use sql


## Day 11 Recap

| What you did | Why it matters |
|---|---|
| Loaded CSVs into SQLite | Data in the real world lives in databases, not files |
| SELECT / WHERE / GROUP BY / ORDER BY | The four keywords that handle 80% of real queries |
| JOIN in SQL | Same logic as pd.merge(), different syntax |
| Math inside SELECT | Computed columns without adding them to the DataFrame first |

### What's coming on Day 12
311 response time analysis — does the city respond faster in wealthier zip codes? You'll use both pandas and SQL to find out.